# Agent 365 - Registry Ingester (Fabric) — recommended default

> **Status: supported (app-only, unattended) — this is the recommended default path** for landing the
> Agent 365 registry into the Lakehouse. As of **PAX `purview-v1.11.11`** and the current
> Microsoft Graph docs, the *Agent 365 catalog* endpoint supports **application permissions** and
> **`/v1.0`**, so this notebook runs **headless on a schedule** with a service principal — no
> interactive sign-in. (It was previously a delegated-only, interactive PREVIEW.)

## What this does

Pulls the tenant's **Agent 365 agent catalogue** (declarative agents, plugins, etc. - including
their data-access **capabilities/permissions**: OneDrive/SharePoint read, Graph connector, code
interpreter, image generation, uploaded files) into the Lakehouse Delta table `dbo.agents_365`
(the table the dashboard reads), keyed on `Title ID` (the `T_`-prefixed titleId).

## Requirements & caveats (read before using)

| Item | Detail |
|---|---|
| **Auth** | **App-only / client credentials** (service principal). Runs unattended. No user sign-in. |
| **Permissions** | **Application** permissions `CopilotPackages.Read.All` **+** `Application.Read.All` **+** `User.Read.All`, **admin-consented**. `User.Read.All` is new - it backs creator resolution tier 1. |
| **Agent 365 licence** | **Still required in the tenant.** This is a *SKU* check, separate from permissions - a missing licence returns **`403`** (`Customer must be licensed for Agent 365`). |
| **Endpoint version** | Uses **`/v1.0`** (GA). Automatically falls back to **`/beta`** if a tenant hasn't surfaced v1.0 yet (PAX itself still calls beta). |
| **Point-in-time only** | No history; deleted agents disappear. `Agent creator` is now resolved to a joinable UPN - see *Creator attribution* below. |

**Relationship to the export lander:** `Copilot_Agent365_Lander.ipynb` (admin-center **export CSV**)
is a **fallback** for tenants that can't grant the app-registration permissions this notebook needs,
or for one-off / evaluation runs. **Prefer this notebook** for scheduled production pipelines — you
get the live capability detail, no CSV upload step, and an automated refresh. The two notebooks are
**alternatives** — they write to the same `dbo.agents_365` table, so running both in the same pipeline
would just clobber each other.

## Setup

- An **app registration** (service principal) with the two **Application** permissions above,
  admin-consented, plus a **client secret** (store it in Key Vault / a Fabric secret, not in code)
  or a **certificate** / **managed identity**.
- Install MSAL only if you use the managed-identity path; the default client-secret path uses
  `requests` and needs nothing extra.

*Endpoint, 28-column schema and capability fields align with the PAX Agent 365 enrichment output so
the dashboard's `Agents 365` query reads them unchanged. Ref: Microsoft Graph
`copilot/admin/catalog/packages` (v1.0) and PAX `purview-v1.11.11`.*

## Creator attribution (`Agent creator UPN`)

`Publisher` is a **display string**, not an identifier - it is length-limited and is not a
usable join key, even though both `Publisher` and `Agent creator` are mapped from it. To
report agents by franchise/function you need a real identity, so this notebook resolves one
and writes two extra columns:

| Column | Contents |
|---|---|
| `Agent creator UPN` | normalised (lowercased, trimmed) UPN - joins directly to `PersonId_Normalized` in `copilot_org_data` |
| `Agent creator source` | which tier resolved it, or `unattributed` |

Three tiers run in order; each only processes agents still unresolved:

| Tier | Method | Permission |
|---|---|---|
| 1 | `ownerId` -> `/users/{id}` via `$batch` | `User.Read.All` **(new)** |
| 2 | `appId` / `agentIdentityId` -> `/servicePrincipals/{id}/owners` | `Application.Read.All` (already required) |
| 3 | earliest Purview audit-log creation event | existing parsed table |

Each tier is independently switchable via the `RESOLVE_VIA_*` flags.

**Always LEFT-join on `Agent creator UPN`.** Agents owned by a service principal or an admin
account resolve to a UPN with no row in the org table, and an inner join would silently drop
them and understate your totals. Surface `Agent creator source` as a slicer so attribution
coverage is visible on the page rather than assumed - a high unattributed rate is an
agent-ownership governance finding, not a bug.

## Optional: full field passthrough

`INCLUDE_RAW_PASSTHROUGH` (default **`False`**) additionally carries **every** field the API
returns, under its native camelCase name, with nested objects serialised to JSON. Fields are
unioned across all agents, so an attribute present on only some agents still becomes a column.

It is **off by default on purpose.** The dashboard's `Agents 365` query is additive - it has
no `Table.SelectColumns` - so every column in `dbo.agents_365` reaches the semantic model.
Turning passthrough on adds roughly 30 unmodelled columns to that table. Enable it when you
need an attribute the canonical schema does not carry, and expect the model to widen.

The canonical columns are never clobbered: where a raw key collides with one, the canonical
value wins, and a raw value that genuinely differs is kept under `<key>_raw`.


## 1. Configuration & app-only sign-in

**App-only (client-credentials)** flow - runs unattended, so this notebook can be a scheduled Fabric
job. No device-code, no browser. The service principal's admin-consented **Application** permissions
(`CopilotPackages.Read.All` + `Application.Read.All`) are carried in the token via the `.default`
scope.

In [ ]:
# === CONFIG ===
TENANT_ID    = '<your-tenant-guid>'
CLIENT_ID    = '<app-reg-client-id>'   # App perms (admin-consented): CopilotPackages.Read.All
                                       #   + Application.Read.All + User.Read.All
TARGET_TABLE = 'dbo.agents_365'
WRITE_MODE   = 'overwrite'
ALLOW_EMPTY_SNAPSHOT = False

# Creator resolution toggles - each tier is independent and skippable.
RESOLVE_VIA_OWNER_ID   = True    # tier 1: ownerId -> /users
RESOLVE_VIA_SP_OWNERS  = True    # tier 2: appId -> /servicePrincipals/{id}/owners
RESOLVE_VIA_AUDIT_LOG  = True    # tier 3: Purview creation events
AUDIT_TABLE            = 'dbo.copilot_interactions_parsed'

# Optional: carry every raw API field through alongside the canonical schema.
# Off by default. The dashboard's `Agents 365` query is additive (no Table.SelectColumns),
# so every column here reaches the semantic model -- enabling this widens it by ~30 columns.
# Turn it on when you need an attribute the canonical schema does not carry.
INCLUDE_RAW_PASSTHROUGH = False

# Client secret - DO NOT hardcode. Pull from Key Vault at runtime, e.g.:
#   CLIENT_SECRET = notebookutils.credentials.getSecret('https://<vault>.vault.azure.net/', 'Agent365AppSecret')
CLIENT_SECRET = '<from-key-vault>'

import requests

def _get_graph_token() -> str:
    url  = f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token'
    data = {
        'client_id':     CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope':         'https://graph.microsoft.com/.default',
        'grant_type':    'client_credentials',
    }
    r = requests.post(url, data=data)
    r.raise_for_status()
    return r.json()['access_token']

TOKEN = _get_graph_token()
print('app-only token acquired:', bool(TOKEN))

## 2. Call the v1.0 catalog endpoint

Unchanged from upstream. Uses `/v1.0` (GA, app-only), auto-falls back to `/beta`.

- **403** = missing Agent 365 licence (SKU check), or no admin consent for `CopilotPackages.Read.All`.
- **401** = token/consent problem, not a code bug.

In [ ]:
import requests

API_VERSION = 'v1.0'


def _base(v):
    return f'https://graph.microsoft.com/{v}/copilot/admin/catalog/packages'


def _validate_catalog_page(data, page_number):
    if not isinstance(data, dict):
        raise ValueError(f'Agent 365 catalog page {page_number} did not return an object.')
    if 'value' not in data:
        raise ValueError(f"Agent 365 catalog page {page_number} is missing required 'value'.")
    value = data['value']
    if not isinstance(value, list):
        raise ValueError(f"Agent 365 catalog page {page_number} returned a non-list 'value'.")
    next_link = data.get('@odata.nextLink')
    if next_link is not None and (not isinstance(next_link, str) or not next_link.strip()):
        raise ValueError(f"Agent 365 catalog page {page_number} returned an invalid '@odata.nextLink'.")
    for item_number, item in enumerate(value, start=1):
        if not isinstance(item, dict):
            raise ValueError(f'Agent 365 catalog page {page_number} item {item_number} is not an object.')
    return value, next_link


BASE = _base(API_VERSION)
H = {'Authorization': f'Bearer {TOKEN}'}

probe = requests.get(f'{BASE}?$top=1', headers=H, timeout=60)
if probe.status_code == 404 and API_VERSION == 'v1.0':
    API_VERSION = 'beta'
    BASE = _base(API_VERSION)
    probe = requests.get(f'{BASE}?$top=1', headers=H, timeout=60)
if probe.status_code == 403:
    raise PermissionError('403: Agent 365 catalog not accessible. Requires an Agent 365 LICENCE in '
                          'the tenant (SKU check) AND admin-consented Application permission '
                          'CopilotPackages.Read.All.')
if probe.status_code == 401:
    raise PermissionError('401: token/consent problem - check admin consent for '
                          'CopilotPackages.Read.All + Application.Read.All + User.Read.All.')
probe.raise_for_status()
print(f'Using {API_VERSION} endpoint.')

packages, url, seen_urls = [], BASE, set()
for page_number in range(1, 501):
    if url in seen_urls:
        raise ValueError(f'Agent 365 catalog paging loop detected at page {page_number}.')
    seen_urls.add(url)
    r = requests.get(url, headers=H, timeout=60)
    r.raise_for_status()
    page, next_url = _validate_catalog_page(r.json(), page_number)
    packages.extend(page)
    url = next_url
    if not url:
        break
else:
    raise ValueError('Agent 365 catalog exceeded the 500-page safety cap.')

print('packages in catalog:', len(packages))

## 3. Fetch per-package detail and flatten every field

The list endpoint returns inventory metadata; the detail endpoint adds usage metrics
and `elementDetails`. We merge both (detail wins on conflict) and flatten the union,
so nothing the API returns is discarded.

In [ ]:
import json as _json


def _stringify(value):
    # Scalars -> str. Nested structures -> compact JSON so nothing is lost.
    if value is None:
        return ''
    if isinstance(value, bool):
        return 'true' if value else 'false'
    if isinstance(value, (dict, list)):
        return _json.dumps(value, ensure_ascii=False, sort_keys=True)
    return str(value)


def _flatten_raw(payload):
    # One flat dict of every top-level API field, native camelCase names preserved.
    flat = {}
    for key, value in payload.items():
        if key.startswith('@odata'):
            continue
        flat[key] = _stringify(value)
    return flat


details = []
for package in packages:
    package_id = package.get('id') or package.get('titleId') or package.get('packageId')
    if package_id in (None, ''):
        raise ValueError('Agent 365 catalog list item is missing id/titleId/packageId.')
    detail_response = requests.get(f'{BASE}/{package_id}', headers=H, timeout=60)
    detail_response.raise_for_status()
    detail = detail_response.json()
    if not isinstance(detail, dict):
        raise ValueError(f'Agent 365 detail for {package_id!r} did not return an object.')
    merged = dict(package)
    merged.update(detail)          # detail wins - it is the richer payload
    details.append(merged)

print(f'details fetched: {len(details)}')

_all_api_fields = sorted({k for d in details for k in d if not k.startswith('@odata')})
print(f'distinct API fields observed: {len(_all_api_fields)}')
print('  ' + ', '.join(_all_api_fields))

## 4. Resolve the agent creator to a joinable UPN

`Publisher` is a truncated display string. This produces `Agent creator UPN`
(lowercased + trimmed, matching the repo's `_normalise_identity`) so it joins
straight to `PersonId_Normalized` in the org table.

Tiers run in order and each only processes agents still unresolved.

In [ ]:
def _normalise_identity(value):
    return (value or '').strip().lower()


# --- Tier 1: ownerId -> /users via $batch -------------------------------------
def _resolve_owner_ids(owner_ids):
    resolved = {}
    owner_ids = [o for o in owner_ids if o]
    for i in range(0, len(owner_ids), 15):
        chunk = owner_ids[i:i + 15]
        batch = {'requests': [
            {'id': str(n), 'method': 'GET',
             'url': f'/users/{oid}?$select=id,userPrincipalName'}
            for n, oid in enumerate(chunk)
        ]}
        resp = requests.post('https://graph.microsoft.com/v1.0/$batch',
                             headers={**H, 'Content-Type': 'application/json'},
                             json=batch, timeout=60)
        if resp.status_code != 200:
            print(f'  tier1: batch failed ({resp.status_code}) - skipping chunk')
            continue
        for item in resp.json().get('responses', []):
            if item.get('status') != 200:
                continue
            body = item.get('body') or {}
            upn = _normalise_identity(body.get('userPrincipalName'))
            if body.get('id') and upn:
                resolved[body['id']] = upn
    return resolved


# --- Tier 2: appId / agentIdentityId -> servicePrincipal owners ---------------
_sp_cache = {}

def _resolve_sp_owner(app_id, agent_identity_id):
    cache_key = (app_id or '', agent_identity_id or '')
    if cache_key in _sp_cache:
        return _sp_cache[cache_key]
    result = ''
    for ident, by_app in ((app_id, True), (agent_identity_id, False)):
        if not ident or result:
            continue
        url = (f"https://graph.microsoft.com/v1.0/servicePrincipals(appId='{ident}')"
               if by_app else
               f'https://graph.microsoft.com/v1.0/servicePrincipals/{ident}')
        sp = requests.get(url, headers=H, timeout=60)
        if sp.status_code != 200:
            continue
        sp_id = (sp.json() or {}).get('id')
        if not sp_id:
            continue
        owners = requests.get(
            f'https://graph.microsoft.com/v1.0/servicePrincipals/{sp_id}/owners'
            '?$select=id,userPrincipalName', headers=H, timeout=60)
        if owners.status_code != 200:
            continue
        for owner in owners.json().get('value', []):
            upn = _normalise_identity(owner.get('userPrincipalName'))
            if upn:
                result = upn
                break
    _sp_cache[cache_key] = result
    return result


# --- Tier 3: Purview audit-log earliest creation event ------------------------
def _audit_creator_map(keys_wanted):
    # Earliest audit event per agent identifier -> creator UPN. Defensive: the
    # parsed table's column names vary by ingester version, so detect rather than
    # assume, and return empty if the shape is not recognisable.
    if not keys_wanted:
        return {}
    try:
        if not spark.catalog.tableExists(AUDIT_TABLE):
            print(f'  tier3: {AUDIT_TABLE} not present - skipping')
            return {}
    except Exception as exc:
        print(f'  tier3: cannot inspect {AUDIT_TABLE} ({exc}) - skipping')
        return {}

    from pyspark.sql import functions as F

    audit = spark.table(AUDIT_TABLE)
    cols = {c.lower(): c for c in audit.columns}

    user_col = next((cols[c] for c in (
        'userid', 'userprincipalname', 'upn', 'userkey', 'createdby',
        'audit_userid_normalized', 'audit_userid') if c in cols), None)
    date_col = next((cols[c] for c in (
        'creationdate', 'createddatetime', 'interactiondate') if c in cols), None)
    agent_col = next((cols[c] for c in (
        'titleid', 'agentid', 'appid', 'copilotagentid', 'source_resourcekey',
        'agent_titleid') if c in cols), None)

    if not (user_col and date_col and agent_col):
        print('  tier3: audit table lacks user/date/agent columns - skipping')
        print(f'         available: {", ".join(sorted(audit.columns))[:300]}')
        return {}

    print(f'  tier3: using {agent_col} / {user_col} / {date_col}')
    # Order on a real timestamp. Casting to string sorts lexicographically, which only
    # coincides with chronological order for ISO-8601 input; anything else silently
    # picks the wrong "earliest" event and so the wrong creator.
    window = audit.select(
        F.lower(F.trim(F.col(agent_col).cast('string'))).alias('k'),
        F.lower(F.trim(F.col(user_col).cast('string'))).alias('upn'),
        F.col(date_col).cast('timestamp').alias('ts'),
    ).where(F.col('k').isNotNull() & (F.col('k') != '') &
            F.col('upn').isNotNull() & (F.col('upn') != '') &
            F.col('ts').isNotNull())

    if window.isEmpty():
        print(f'  tier3: no rows with a parseable {date_col}; skipping')
        return {}

    earliest = (window.groupBy('k')
                      .agg(F.min(F.struct('ts', 'upn')).alias('first'))
                      .select('k', F.col('first.upn').alias('upn')))

    wanted = {k.lower() for k in keys_wanted if k}
    return {r['k']: r['upn'] for r in earliest.collect() if r['k'] in wanted}

In [ ]:
# === RESOLVE CREATORS =========================================================
creator_upn    = {}   # index in `details` -> upn
creator_source = {}   # index in `details` -> which tier resolved it

def _detail_title_id(detail):
    raw = detail.get('id') or detail.get('titleId') or detail.get('packageId')
    if raw in (None, ''):
        raise ValueError('Agent 365 detail is missing id/titleId/packageId.')
    raw = str(raw)
    return raw if raw.startswith(('T_', 'P_')) else f'T_{raw}'

title_ids = [_detail_title_id(d) for d in details]

# Tier 1
if RESOLVE_VIA_OWNER_ID:
    owner_ids = [d.get('ownerId') or '' for d in details]
    populated = sum(1 for o in owner_ids if o)
    print(f'tier1: ownerId populated on {populated}/{len(details)} agents')
    if populated:
        upn_map = _resolve_owner_ids(sorted({o for o in owner_ids if o}))
        for idx, oid in enumerate(owner_ids):
            upn = upn_map.get(oid, '')
            if upn:
                creator_upn[idx], creator_source[idx] = upn, 'ownerId'
    print(f'tier1: resolved {len(creator_upn)}')

# Tier 2
if RESOLVE_VIA_SP_OWNERS:
    before = len(creator_upn)
    for idx, detail in enumerate(details):
        if idx in creator_upn:
            continue
        upn = _resolve_sp_owner(detail.get('appId'), detail.get('agentIdentityId'))
        if upn:
            creator_upn[idx], creator_source[idx] = upn, 'servicePrincipalOwner'
    print(f'tier2: resolved {len(creator_upn) - before} more')

# Tier 3
if RESOLVE_VIA_AUDIT_LOG:
    before = len(creator_upn)
    unresolved_keys = set()
    for idx, detail in enumerate(details):
        if idx in creator_upn:
            continue
        for candidate in (title_ids[idx], detail.get('appId'), detail.get('agentIdentityId')):
            if candidate:
                unresolved_keys.add(str(candidate))
    audit_map = _audit_creator_map(unresolved_keys)
    if audit_map:
        for idx, detail in enumerate(details):
            if idx in creator_upn:
                continue
            for candidate in (title_ids[idx], detail.get('appId'), detail.get('agentIdentityId')):
                upn = audit_map.get(str(candidate or '').lower(), '')
                if upn:
                    creator_upn[idx], creator_source[idx] = upn, 'auditLog'
                    break
    print(f'tier3: resolved {len(creator_upn) - before} more')

resolved = len(creator_upn)
print(f'\ncreator resolution: {resolved}/{len(details)} '
      f'({resolved / len(details) * 100:.1f}%)' if details else 'no agents')
for tier in ('ownerId', 'servicePrincipalOwner', 'auditLog'):
    n = sum(1 for v in creator_source.values() if v == tier)
    if n:
        print(f'   {tier:24} {n}')
unattributed = len(details) - resolved
if unattributed:
    print(f'   {"UNATTRIBUTED":24} {unattributed}  <- left-join these, do not drop them')

## 5. Shape -> canonical schema + full passthrough -> Delta

The 37 `CANONICAL` columns keep their exact upstream names and mappings so the PBIT
reads them unchanged. Every raw API field is then appended under its native name,
plus the two new creator columns.

In [ ]:
from pyspark.sql import functions as F


def _join(value):
    if isinstance(value, list):
        return ';'.join(
            _json.dumps(item, ensure_ascii=False) if isinstance(item, (dict, list)) else str(item)
            for item in value if item is not None
        )
    return '' if value is None else str(value)


def _access(value):
    if value is None:
        return ''
    if not isinstance(value, list):
        raise ValueError('Agent 365 access collections must be lists when present.')
    out = []
    for item in value:
        if isinstance(item, dict):
            out.append(item.get('displayName') or item.get('id') or _json.dumps(item, ensure_ascii=False))
        else:
            out.append(str(item))
    return ';'.join(out)


# elementDetails carries vendor-supplied payloads that vary across publishers.
# A single malformed entry must not discard an entire tenant snapshot, so structural
# problems are counted and skipped; the census is printed after shaping.
_ELEMENT_SKIPS = {}


def _note_element_skip(reason):
    _ELEMENT_SKIPS[reason] = _ELEMENT_SKIPS.get(reason, 0) + 1


def _elements(detail):
    raw_groups = detail.get('elementDetails')
    if raw_groups is None:
        return '', '', ''
    if not isinstance(raw_groups, list):
        _note_element_skip('elementDetails is not a list')
        return '', '', ''
    types, bots, commands = [], [], []
    for group in raw_groups:
        if not isinstance(group, dict):
            _note_element_skip('group is not an object')
            continue
        element_type = group.get('elementType')
        if element_type:
            types.append(str(element_type))
        elements = group.get('elements') or []
        if not isinstance(elements, list):
            _note_element_skip('elements is not a list')
            continue
        for element in elements:
            if not isinstance(element, dict):
                _note_element_skip('element is not an object')
                continue
            raw_definition = element.get('definition')
            if raw_definition in (None, ''):
                continue
            if isinstance(raw_definition, str):
                try:
                    definition = _json.loads(raw_definition)
                except _json.JSONDecodeError:
                    _note_element_skip('definition is not valid JSON')
                    continue
            else:
                definition = raw_definition
            if not isinstance(definition, dict):
                _note_element_skip('definition did not decode to an object')
                continue
            if definition.get('botId'):
                bots.append(str(definition['botId']))
            for command_list in definition.get('commandLists') or []:
                if not isinstance(command_list, dict):
                    _note_element_skip('commandLists entry is not an object')
                    continue
                for command in command_list.get('commands') or []:
                    if isinstance(command, dict) and command.get('title'):
                        commands.append(str(command['title']))
            for command in definition.get('commands') or []:
                if isinstance(command, dict) and command.get('title'):
                    commands.append(str(command['title']))
    return (
        ';'.join(dict.fromkeys(types)),
        ';'.join(dict.fromkeys(bots)),
        ';'.join(dict.fromkeys(commands)),
    )


def _merge_raw_passthrough(row, raw):
    """Add every raw API field to `row` without creating a case-insensitive duplicate.

    Spark resolves column names case-insensitively (spark.sql.caseSensitive=false), so a
    raw key that differs from a canonical column only by case -- 'version' vs 'Version',
    'publisher' vs 'Publisher' -- cannot coexist with it. Keeping both fails the frame
    with AMBIGUOUS_REFERENCE on select and COLUMN_ALREADY_EXISTS on the Delta write.

    An exact-name match is already mapped, so it is skipped. For a case-only clash:
      - same value      -> skip; the canonical column already carries it
      - differing value -> keep under '<key>_raw' so the passthrough guarantee holds
    """
    folded = {key.casefold(): key for key in row}
    for api_key, api_value in raw.items():
        clash = folded.get(api_key.casefold())
        if clash is None:
            row[api_key] = api_value
            folded[api_key.casefold()] = api_key
            continue
        if str(row[clash]) == str(api_value):
            continue                       # canonical column is identical; nothing lost
        alt, suffix = f'{api_key}_raw', 2
        while alt.casefold() in folded:
            alt, suffix = f'{api_key}_raw{suffix}', suffix + 1
        row[alt] = api_value
        folded[alt.casefold()] = alt
    return row


rows = []
for idx, detail in enumerate(details):
    element_types, bot_ids, commands = _elements(detail)
    row = {
        # --- canonical (unchanged names/mappings, PBIT-compatible) ------------
        'Agent name':         detail.get('displayName') or '',
        'Title ID':           title_ids[idx],
        'Version':            detail.get('version') or '',
        'Entra Agent ID':     detail.get('agentIdentityId') or '',
        'Bot Id':             bot_ids,
        'App Id':             detail.get('appId') or '',
        'Asset Id':           detail.get('assetId') or '',
        'Publisher':          detail.get('publisher') or '',
        'Agent creator':      detail.get('publisher') or '',
        'Agent creator ID':   detail.get('ownerId') or '',
        'Agent type (A365)':  detail.get('type') or '',
        'Created in':         detail.get('platform') or '',
        'Date created':       detail.get('createdDateTime') or '',
        'Last updated':       detail.get('lastModifiedDateTime') or '',
        'Agent description':  detail.get('shortDescription') or detail.get('longDescription') or '',
        'Categories':         _join(detail.get('categories')),
        'Supported in':       _join(detail.get('supportedHosts')),
        'Availability':       detail.get('availableTo') or '',
        'Status':             detail.get('deployedTo') or '',
        'Is Blocked':         str(detail.get('isBlocked', '') or ''),
        'Users shared':       _access(detail.get('sharedWithUsersAndGroups')),
        'Groups shared':      _access(detail.get('allowedUsersAndGroups')),
        'Element types':      element_types or _join(detail.get('elementTypes')),
        'Custom actions':     commands,
        'Active Users':       str(detail.get('activeUsers', '') or ''),
        'Total sessions':     str(detail.get('totalSessions', '') or ''),
        'Exception rate':     str(detail.get('exceptionRate', '') or ''),
        'Run Time':           str(detail.get('totalRunTimeInHours', '') or ''),
        'Last Activity Date': detail.get('lastUsedDateTime') or '',

        # --- NEW: joinable creator identity ----------------------------------
        'Agent creator UPN':    creator_upn.get(idx, ''),
        'Agent creator source': creator_source.get(idx, 'unattributed'),
    }

    # --- NEW: full raw passthrough -------------------------------------------
    if INCLUDE_RAW_PASSTHROUGH:
        _merge_raw_passthrough(row, _flatten_raw(detail))

    rows.append(row)


def _normalise_registry_key(value):
    return (value or '').strip().upper()


def _dedupe_registry_rows(rows):
    seen, deduped = {}, []
    for row in rows:
        key = _normalise_registry_key(row.get('Title ID'))
        if not key:
            raise ValueError('Agent 365 row is missing Title ID; refusing to write an ambiguous snapshot.')
        comparable = {c: '' if v is None else str(v).strip() for c, v in row.items()}
        prior = seen.get(key)
        if prior is None:
            seen[key] = comparable
            deduped.append(row)
            continue
        if comparable != prior:
            raise ValueError(f'Conflicting Agent 365 rows detected for {key!r}; refusing to overwrite a good snapshot.')
    return deduped


def _guard_registry_snapshot(rows, table_name):
    exists = bool(spark.catalog.tableExists(table_name))
    if not rows and exists:
        raise ValueError(f'Fetched 0 Agent 365 rows; refusing to replace existing {table_name}.')
    if not rows and not ALLOW_EMPTY_SNAPSHOT:
        raise ValueError(
            f'Fetched 0 Agent 365 rows and {table_name} does not exist yet. '
            'Set ALLOW_EMPTY_SNAPSHOT = True only for an intentional empty first install.'
        )


rows = _dedupe_registry_rows(rows)
_guard_registry_snapshot(rows, TARGET_TABLE)

rows = [{k: ('' if v is None else str(v)) for k, v in row.items()} for row in rows]

CANONICAL = [
    'Agent name', 'Supported in', 'Date created', 'Agent creator', 'Publisher',
    'Agent type (A365)', 'Version', 'Availability', 'Agent creator ID',
    'Agent description', 'Created in', 'Last updated', 'Custom actions',
    'Title ID', 'Sensitivity',
    'Can read OneDrive and Sharepoint items', 'OneDrive and Sharepoint items',
    'Can read OneDrive files', 'OneDrive files', 'OneDrive sites',
    'Can read Sharepoint sites and files', 'Sharepoint files', 'Sharepoint sites',
    'Can extend to Graph connector', 'Graph connector details',
    'Can generate images using user prompt', 'Can use code interpreter',
    'Contains uploaded files', 'Uploaded files', 'Status',
    'Active Users', 'Total sessions', 'Exception rate', 'Last Activity Date',
    'Deployment', 'Run Time', 'Risks',
]
NEW_CANONICAL = ['Agent creator UPN', 'Agent creator source']

# Union every key across every row so schema is stable even when the API returns
# different attributes per agent.
all_keys = []
for row in rows:
    for k in row:
        if k not in all_keys:
            all_keys.append(k)
for column in CANONICAL + NEW_CANONICAL:
    if column not in all_keys:
        all_keys.append(column)
for row in rows:
    for column in all_keys:
        row.setdefault(column, '')

ordered = (CANONICAL + NEW_CANONICAL
           + [c for c in all_keys if c not in CANONICAL and c not in NEW_CANONICAL])

if rows:
    df = spark.createDataFrame(rows).select(*[F.col(f'`{c}`') for c in ordered])
else:
    df = spark.createDataFrame([], ','.join(f'`{c}` string' for c in ordered))

filled = {c: sum(1 for r in rows if r.get(c)) for c in ordered}
print(f'packages shaped : {len(rows)}')
print(f'total columns   : {len(ordered)}  '
      f'(canonical {len(CANONICAL)} + new {len(NEW_CANONICAL)} + passthrough '
      f'{len(ordered) - len(CANONICAL) - len(NEW_CANONICAL)})')
print('\npopulated columns:')
for c in ordered:
    if filled.get(c):
        print(f'   {c:44} {filled[c]}/{len(rows)}')
blank = [c for c in ordered if not filled.get(c)]
if blank:
    print(f'\nblank ({len(blank)}): {", ".join(blank)}')
    print('   -> observability-only fields are populated by the Admin Center CSV export only.')
if _ELEMENT_SKIPS:
    print(f'\nelementDetails entries skipped ({sum(_ELEMENT_SKIPS.values())}):')
    for _reason in sorted(_ELEMENT_SKIPS):
        print(f'   {_reason:44} {_ELEMENT_SKIPS[_reason]}')
    print('   -> malformed publisher payloads; set INCLUDE_RAW_PASSTHROUGH to retain them verbatim.')

(df.write.mode(WRITE_MODE)
   .option('overwriteSchema', 'true')
   .option('delta.columnMapping.mode', 'name')
   .option('delta.minReaderVersion', '2')
   .option('delta.minWriterVersion', '5')
   .format('delta').saveAsTable(TARGET_TABLE))
print(f'\nwrote -> {TARGET_TABLE} ({WRITE_MODE}) | columns: {len(df.columns)}')

## 6. Downstream wiring

**Lakehouse join** - `Agent creator UPN` is already lowercased/trimmed, so it matches
`PersonId_Normalized` directly:

```sql
SELECT a.[Title ID], a.[Agent name], a.[Agent creator source],
       COALESCE(o.Level3_Name, 'Unattributed') AS Franchise,
       o.JobTitle
FROM   dbo.agents_365 a
LEFT   JOIN dbo.copilot_org_data o
       ON a.[Agent creator UPN] = o.PersonId_Normalized;
```

**LEFT join, always.** An inner join silently drops every unattributed agent and
understates your totals.

**Power BI model** - relationship `agents_365[Agent creator UPN]` -> `org[PersonId_Normalized]`,
many-to-one, single direction. Set `Agent creator source` as a slicer so coverage is
visible on the page rather than assumed.

**Franchise from Level1-14** - these are positional, not semantic. `Level3_Name` is a
placeholder above: a person six hops from the CEO and one eleven hops down have their
franchise at different levels. Prefer Graph `companyName`/`department`, or build a
`franchise_lookup` mapping table and `COALESCE` across levels to find the first match.